In [17]:
import os
import pandas as pd

artemis_data = [x for x in os.listdir("humanData/") if x.startswith("PlausibleSpecs")]
artemis_data


['PlausibleSpecs(2).xlsx',
 'PlausibleSpecs(1).xlsx',
 'PlausibleSpecs.xlsx',
 'PlausibleSpecs(4).xlsx']

In [18]:
from openpyxl import load_workbook
import json
import random
import csv
import os
import sys
import re
import pandas as pd
import spot 


def extract_ap_mapping(ltl_formula: str) -> str:
    """
    Extract AP names from the GT formula using Spot.
    Return them as a set of strings (stringified for the prompt).
    """
    f = spot.formula(ltl_formula)
    aps = sorted(str(ap) for ap in spot.atomic_prop_collect(f))

    return "{" + ", ".join(aps) + "}"

In [ ]:



dataset = []
for inp in artemis_data:
    wb = load_workbook("humanData/"+inp, data_only=True)
    print(inp)
    ws = wb.active

    # ------------------------------------------------------------
    # Detect NL column dynamically from header row
    # ------------------------------------------------------------
    header_row = next(ws.iter_rows(min_row=1, max_row=1, values_only=True))

    nl_col_idx = None

    for idx, value in enumerate(header_row):
        if value == "NL":
            nl_col_idx = idx
            break

    if nl_col_idx is None:
        raise ValueError('Could not find header column named "NL"')

    # Relative offsets from NL column
    OFFSET_F = 4   # decision3
    OFFSET_G = 5   # bool_expr3 / bool_expr4
    OFFSET_H = 6   # decision2
    OFFSET_I = 7   # bool_expr2
    OFFSET_J = 8   # decision1
    OFFSET_K = 9   # bool_expr1

    pairs = []
    malformed_count = 0

    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):

        try:
            nl_text = row[nl_col_idx].value  # Column B

            # ------------------------------------------------------------
            # STEP 1-3 : decision2 / bool_expr2
            # ------------------------------------------------------------
            ltl_prefix = None
            used_bool_expr2 = False
            bool_expr2 = None

            h_val = row[nl_col_idx + OFFSET_H].value  # Column H

            if h_val:
                h_json = json.loads(h_val)
                decision2 = h_json[0]["decision2"]

                if "upon bool_exp2" in decision2[0]:
                    i_val = row[nl_col_idx + OFFSET_I].value  # Column I

                    if not i_val:
                        raise ValueError("Missing column I")

                    i_json = json.loads(i_val)

                    bool_expr2 = i_json[0]["bool_exp2"][0]
                    used_bool_expr2 = True

            # ------------------------------------------------------------
            # STEP 4-6 : decision1 / bool_expr1
            # ------------------------------------------------------------
            j_val = row[nl_col_idx + OFFSET_J].value  # Column J

            if not j_val:
                raise ValueError("Missing column J")

            j_json = json.loads(j_val)
            decision1 = j_json[0]["decision1"]

            bool_expr1 = None

            two_brackets = False

            if "while bool_exp1" in decision1[0] or "whenever bool_exp1" in decision1[0]:
                k_val = row[nl_col_idx + OFFSET_K].value  # Column K

                if not k_val:
                    raise ValueError("Missing column K")

                k_json = json.loads(k_val)
                bool_expr1 = k_json[0]["bool_exp1"][0]

                if used_bool_expr2:
                    ltl_prefix = (
                        f"!{bool_expr2} U ({bool_expr2} & G({bool_expr1} -> "
                    )
                    two_brackets = True
                else:
                    ltl_prefix = f"G({bool_expr1} -> "

            else:

                if "upon bool_exp1" in decision1[0]:

                    k_val = row[nl_col_idx + OFFSET_K].value  # Column K

                    if not k_val:
                        raise ValueError("Missing column K")

                    k_json = json.loads(k_val)
                    bool_expr1 = k_json[0]["bool_exp1"][0]

                    ltl_prefix = (
                        f"!{bool_expr1} U ({bool_expr1} & "
                    )

                elif used_bool_expr2:
                    ltl_prefix = (
                        f"!{bool_expr2} U ({bool_expr2} & "
                    )
                else:

                    ltl_prefix = "("

                    # raise ValueError("No usable decision1/decision2 logic")

            # ------------------------------------------------------------
            # STEP 7-8 : decision3 random selection
            # ------------------------------------------------------------
            f_val = row[nl_col_idx + OFFSET_F].value  # Column F

            if not f_val:
                raise ValueError("Missing column F")

            f_json = json.loads(f_val)
            decision3 = f_json[0]["decision3"]

            valid_choices = [
                s for s in decision3
                if "N_DURATION" not in s
            ]

            if not valid_choices:
                raise ValueError("No valid decision3 entries")

            chosen = random.choice(valid_choices)

            # ------------------------------------------------------------
            # STEP 9 : bool_expr3 / bool_expr4
            # ------------------------------------------------------------
            g_val = row[nl_col_idx + OFFSET_G].value  # Column G

            if not g_val:
                raise ValueError("Missing column G")

            g_json = json.loads(g_val)
            g_first = g_json[0]

            bool_expr3 = g_first["bool_exp3"][0]

            # ------------------------------------------------------------
            # STEP 10-11 : finish formula
            # ------------------------------------------------------------
            suffix = None

            if chosen == "eventually satisfy bool_exp3":
                suffix = f"F({bool_expr3})"

            elif chosen == "always satisfy bool_exp3":
                suffix = f"G({bool_expr3})"

            elif chosen == "at the next timepoint satisfy bool_exp3":
                suffix = f"X({bool_expr3})"

            elif chosen == "immediately satisfy bool_exp3":
                suffix = f"({bool_expr3})"

            elif chosen == "until bool_exp4, satisfy bool_exp3":
                bool_expr4 = g_first["bool_exp4"][0]
                suffix = f"({bool_expr3} U {bool_expr4})"

            else:
                raise ValueError(f"Unsupported decision3 option: {chosen}")

            ltl_formula = ltl_prefix + suffix + ")"
            if two_brackets:
                ltl_formula += ")"

            try:
                ap_set = extract_ap_mapping(ltl_formula)
            except Exception as exc:
                print(
                    f"Skipping item {row_idx} due to Spot parse error:\n"
                    f"  Formula: {ltl_formula}\n"
                    f"  Error:   {exc}",
                    file=sys.stderr,
                )
                parse_errors += 1
                break

            dataset.append((nl_text, ltl_formula, ap_set))
        except Exception as e:
            print(f"Malformed item {idx}: {e}")


PlausibleSpecs(2).xlsx
Malformed item 0: No valid decision3 entries
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
PlausibleSpecs(1).xlsx
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
Malformed item 0: Missing column J
PlausibleSpecs.xlsx
PlausibleSpecs(4).xlsx
Malformed item 1: Unsupported decision3 option: before bool_exp4, satisfy bool_exp3
Malformed item 1: No valid decision3 entries
Malformed item 1: Unsupported decisio

In [ ]:
with open("HumanDataProcessed/ARTEMIS_processed_dataset.csv", "w", newline="") as csvfile:
    writer = csv.DictWriter(
        csvfile,
        fieldnames=["Requirement", "Ground Truth","Atomic Proposition"],
    )
    writer.writeheader()
        
    for nl_text, ltl_formula, ap_set in dataset:
        writer.writerow({
            'Requirement': nl_text,
            'Ground Truth': ltl_formula,
            'Atomic Proposition': ap_set,
        })


In [10]:
dataset = []
import json
import re
with open("humanData/spacewire.json", "r", encoding="utf-8") as f:
    data = json.load(f)

if not isinstance(data, list):
    raise ValueError("Input JSON must contain an array of objects.")


for idx, item in enumerate(data):
    requirement = str(item.get("text", "")).strip()

    logic_entries = item.get("logics", [])

    if not isinstance(logic_entries, list):
        continue

    found_ltl = False

    for logic in logic_entries:
        if logic.get("type") != "LTL":
            continue

        found_ltl = True

        f_code = logic.get("f_code", "")

        if not f_code or not str(f_code).strip():
            break

        ground_truth = str(f_code).strip()

        ground_truth = ground_truth.replace("-->", "->")
        ground_truth = re.sub(r"\bnot\b", "!", ground_truth)
        ground_truth = re.sub(r"\band\b", "&", ground_truth)
        ground_truth = re.sub(r"\bor\b", "&", ground_truth)

        try:
            ap_set = extract_ap_mapping(ground_truth)
        except Exception as exc:
            print(
                f"Skipping item {idx} due to Spot parse error:\n"
                f"  Formula: {ground_truth}\n"
                f"  Error:   {exc}",
                file=sys.stderr,
            )

            break

        dataset.append((requirement, ground_truth, ap_set))

        # only take the first valid LTL formula
        break

    if not found_ltl:
        continue

Skipping item 43 due to Spot parse error:
  Formula: G((transmit_credit ==0 ) -> ((stop_sending_NChar) U (transmit_credit == 8))) & G((transmit_credit == 0 ) -> ( send_FCT & ( send_Null & send_codes)) )
  Error:   
>>> G((transmit_credit ==0 ) -> ((stop_sending_NChar) U (transmit_credit == 8))) & G((transmit_credit == 0 ) -> ( send_FCT & ( send_Null & send_codes)) )
                       ^
syntax error, unexpected invalid token

>>> G((transmit_credit ==0 ) -> ((stop_sending_NChar) U (transmit_credit == 8))) & G((transmit_credit == 0 ) -> ( send_FCT & ( send_Null & send_codes)) )
                       ^^^
ignoring this

>>> G((transmit_credit ==0 ) -> ((stop_sending_NChar) U (transmit_credit == 8))) & G((transmit_credit == 0 ) -> ( send_FCT & ( send_Null & send_codes)) )
                                                                         ^
syntax error, unexpected invalid token

>>> G((transmit_credit ==0 ) -> ((stop_sending_NChar) U (transmit_credit == 8))) & G((transmit_credit

In [12]:



with open("HumanDataProcessed/SPACEWIRE_processed_dataset.csv", "w", newline="") as csvfile:
    writer = csv.DictWriter(
        csvfile,
        fieldnames=["Requirement", "Ground Truth","Atomic Proposition"],
    )
    writer.writeheader()
        
    for nl_text, ltl_formula, ap_set in dataset:
        writer.writerow({
            'Requirement': nl_text,
            'Ground Truth': ltl_formula,
            'Atomic Proposition': ap_set,
        })

In [ ]:
data

In [22]:

dataset = []
def normalize_ltl_variables(formula):
    """
    Aggressive normalization of variable names and operators.
    """

    # ------------------------------------------------------------
    # Normalize temporal operators
    # ------------------------------------------------------------
    formula = formula.replace("<>", "F ")
    formula = formula.replace("[]", "G ")

    formula = formula.replace("AG", "G ")
    formula = formula.replace("AF", "F ")

    formula = formula.replace("EG", "G ")
    formula = formula.replace("EF", "F ")

    # Convert:
    # A[exp1 U exp2]  -->  exp1 U exp2
    formula = re.sub(r"A\[(.*?)\]", r"\1", formula)
    formula = re.sub(r"E\[(.*?)\]", r"\1", formula)

    # ------------------------------------------------------------
    # Normalize boolean operators
    # ------------------------------------------------------------
    formula = formula.replace("||", "|")
    formula = formula.replace("&&", "&")

    def replace_function_var(match):
        func_name = match.group(1)
        args = match.group(2)

        # Split args by comma and normalize
        parts = [p.strip() for p in args.split(",")]

        joined = "_".join(parts)

        return f"{func_name}_{joined}"

    formula = re.sub(
        r"\b([A-Za-z_][A-Za-z0-9_]*)\(([^()]+)\)",
        replace_function_var,
        formula,
    )

    formula = formula.replace(".", "_")
    formula = formula.replace("=", "_")
    formula = formula.replace("+", "_")

    formula = formula.replace("[", "_")
    formula = formula.replace("]", "")
    formula = formula.replace(":", "_")

    # Collapse repeated underscores
    formula = re.sub(r"_+", "_", formula)


    # ------------------------------------------------------------
    # Repair malformed parentheses
    #
    # Removes unmatched closing parens.
    # Adds missing closing parens at end.
    # ------------------------------------------------------------
    repaired = []
    open_count = 0

    for ch in formula:

        if ch == "(":
            open_count += 1
            repaired.append(ch)

        elif ch == ")":
            if open_count > 0:
                open_count -= 1
                repaired.append(ch)
            else:
                # Skip unmatched closing paren
                continue
        else:
            repaired.append(ch)

    # Add missing closing parentheses
    repaired.append(")" * open_count)

    formula = "".join(repaired)


    # Remove spaces around underscores
    formula = re.sub(r"\s*_\s*", "_", formula)

    return formula.strip()



FIELD_NAMES = {
    "REQUIREMENT:",
    "REFINEMENT:",
    "PATTERN:",
    "SCOPE:",
    "PARAMETERS:",
    "LTL:",
    "CTL:",
    "NOTE:",
    "SOURCE:",
    "DOMAIN:",
    "ORIGINAL",
    "REWRITING",
    "NOTE"
}

def starts_with_field(line):
    return any(line.startswith(field) for field in FIELD_NAMES)




with open("humanData/ALL.txt", "r", encoding="utf-8") as f:
    content = f.read()

# Split entries by blank lines
# (works for typical structured requirement datasets)
raw_items = re.split(r"\n\s*\n", content)

malformed_count = 0

for idx, item in enumerate(raw_items, start=1):

    try:
        requirement = None
        refinement = None
        ltl = None

        lines = item.splitlines()

        i = 0

        while i < len(lines):

            line = lines[i].strip()

            # --------------------------------------------------------
            # REQUIREMENT
            # --------------------------------------------------------
            if line.startswith("REQUIREMENT:"):
                requirement = line[len("REQUIREMENT:"):].strip()

            # --------------------------------------------------------
            # REFINEMENT
            # --------------------------------------------------------
            elif line.startswith("REFINEMENT:"):
                refinement = line[len("REFINEMENT:"):].strip()

            # --------------------------------------------------------
            # LTL (possibly multiline)
            # --------------------------------------------------------
            elif line.startswith("LTL:"):

                ltl_lines = [
                    line[len("LTL:"):].strip()
                ]

                i += 1

                while i < len(lines):

                    next_line = lines[i].strip()

                    # Stop if another field begins
                    if starts_with_field(next_line):
                        i -= 1
                        break

                    if next_line:
                        ltl_lines.append(next_line)

                    i += 1

                ltl = " ".join(ltl_lines)

            i += 1

        # ------------------------------------------------------------
        # Validate required fields
        # ------------------------------------------------------------
        if not requirement:
            raise ValueError("Missing REQUIREMENT")

        if not ltl:
            raise ValueError("Missing LTL")

        # ------------------------------------------------------------
        # Combine REQUIREMENT + REFINEMENT
        # ------------------------------------------------------------
        nl_text = requirement

        if refinement:
            nl_text += f" ({refinement})"

        # ------------------------------------------------------------
        # Normalize LTL syntax
        # ------------------------------------------------------------
        ltl = normalize_ltl_variables(ltl)

        try:
            ap_set = extract_ap_mapping(ltl)
        except Exception as exc:
            print(
                f"Skipping item {idx} due to Spot parse error:\n"
                f"  Formula: {ltl}\n"
                f"  Error:   {exc}",
                file=sys.stderr,
            )
            print(lines)
            continue

        dataset.append((nl_text, ltl, ap_set))

    except Exception as e:
        
        if "Missing LTL" not in str(e):
            print(f"Malformed item {idx}: {e}")
            print(lines)


    

['REQUIREMENT: If enqueue(d1) then invocation of forward iterator ', '             will call Process_type(d1)', 'REFINEMENT: If between an enqueue of d1 and the initiation of a forward ', '            iteration, d1 is not dequeued, then it will eventually be ', '            produced by the iteration.', 'PATTERN: Constrained Response-chain 2-1', 'SCOPE: Global', 'PARAMETERS: Call events with parameters', 'LTL: [](call_Enqueue(d1) & (!return_Dequeue(d1) U call_Top_Down) ->', '        <>(call_Top_Down & <>call_P(d1,*)))', 'NOTE: Since these are all events we can drop the nexts', 'SOURCE: Matthew Dwyer ', 'DOMAIN: Generic Container ADTs']
['REQUIREMENT: If enqueue(d1) then invocation of backward iterator ', '             will call Process_type(d1)', 'REFINEMENT: If between an enqueue of d1 and the initiation of a backward ', '            iteration, d1 is not dequeued, then it will eventually be ', '            produced by the iteration.', 'PATTERN: Constrained Response-chain 2-1', 'SCOPE: 

Skipping item 6 due to Spot parse error:
  Formula: G (call_Enqueue_d1 & (!return_Dequeue_d1 U call_Top_Down) -> F (call_Top_Down & F call_P_d1_*))
  Error:   
>>> G (call_Enqueue_d1 & (!return_Dequeue_d1 U call_Top_Down) -> F (call_Top_Down & F call_P_d1_*))
                                                                                                 ^
syntax error, unexpected closing parenthesis

>>> G (call_Enqueue_d1 & (!return_Dequeue_d1 U call_Top_Down) -> F (call_Top_Down & F call_P_d1_*))
                                                                                                ^
missing right operand for "and operator"


Skipping item 7 due to Spot parse error:
  Formula: G (call_Enqueue_d1 & (!return_Dequeue_d1 U call_Bottom_Up) -> F (call_Bottom_Up & F call_P_d1_*))
  Error:   
>>> G (call_Enqueue_d1 & (!return_Dequeue_d1 U call_Bottom_Up) -> F (call_Bottom_Up & F call_P_d1_*))
                                                                                          

In [23]:


with open("HumanDataProcessed/DWYER_processed_dataset.csv", "w", newline="") as csvfile:
    writer = csv.DictWriter(
        csvfile,
        fieldnames=["Requirement", "Ground Truth","Atomic Proposition"],
    )
    writer.writeheader()
        
    for nl_text, ltl_formula, ap_set in dataset:
        writer.writerow({
            'Requirement': nl_text,
            'Ground Truth': ltl_formula,
            'Atomic Proposition': ap_set,
        })

In [24]:
len(dataset)

101

In [10]:
print(len(dataset))

101


In [12]:
e

NameError: name 'e' is not defined